# Sprint 3 — Comparación y selección de candidatos

    Selecciona candidatos para tuning. La selección es reproducible y queda guardada.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.io_utils import load_kick_data, save_df
from src.preprocessing import prepare_features, split_X_y

pd.set_option('display.max_columns', 120)
print('Proyecto:', PROJECT_ROOT)

Proyecto: c:\Users\PONCE\dp261-g1


In [2]:
results = pd.read_csv(REPORTS_DIR / "baseline_cv_results.csv")

# Definimos un umbral de estabilidad (Gap máximo de 15%)
MAX_RECALL_GAP = 0.15

ok = results[
    results["status"].eq("ok") & 
    ~results["model"].str.contains("Dummy", case=False, na=False) &
    (results["recall_gap"] < MAX_RECALL_GAP) # <-- FILTRO CRÍTICO
].copy()

candidates = (
    ok
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False]
    )
    .head(3)
    .copy()
)

candidates["selected_for_tuning"] = True

display(candidates[[
    "model",
    "recall_cv_mean",
    "f2_cv_mean",
    "precision_cv_mean",
    "f1_cv_mean",
    "f05_cv_mean",
    "roc_auc_cv_mean",
    "average_precision_cv_mean",
    "fit_time_mean",
    "selected_for_tuning"
]])

candidates.to_csv(
    REPORTS_DIR / "model_selection_candidates.csv",
    index=False
)

,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,roc_auc_cv_mean,average_precision_cv_mean,fit_time_mean,selected_for_tuning
1,GradientBoosting,0.246748,0.287207,0.836724,0.380922,0.565646,0.760120,0.467894,8.583468,True
2,HistGradientBoosting,0.239837,0.280418,0.870166,0.375832,0.569916,0.760755,0.471401,0.505993,True


VILLANOS

In [5]:
# 1. Creamos discard comparando contra TODOS los modelos originales (results)
# pero quitando los que sí seleccionamos como candidatos
discard = results[
    ~results["model"].isin(candidates["model"]) & 
    ~results["model"].str.contains("Dummy", case=False, na=False)
].copy()

f2_median = results["f2_cv_mean"].median()
recall_median = results["recall_cv_mean"].median()

# 2. Corregimos el orden de la lógica (np.select se detiene en la primera que cumple)
discard["discard_reason"] = np.select(
    [
        discard["recall_gap"] > 0.20,            # Prioridad 1: Overfitting
        discard["recall_cv_mean"].fillna(0) == 0, # Prioridad 2: No funciona
        discard["f2_cv_mean"].fillna(0) < f2_median,
        discard["recall_cv_mean"].fillna(0) < recall_median,
    ],
    [
        "Overfitting crítico (Gap > 20%)",
        "No detecta Bad Buys o recall nulo",
        "Menor F2 que los modelos priorizados",
        "Menor recall que los modelos priorizados",
    ],
    default="Menor prioridad para tuning"
)

display(discard[[
    "model",
    "recall_cv_mean",
    "f2_cv_mean",
    "precision_cv_mean",
    "discard_reason"
]])

,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,discard_reason
0,DecisionTree,0.347967,0.338472,0.305497,Overfitting crítico (Gap > 20%)
3,RandomForest,0.232520,0.272730,0.886383,Overfitting crítico (Gap > 20%)
5,LogisticRegression,NaN,NaN,NaN,No detecta Bad Buys o recall nulo
6,LinearSVM,NaN,NaN,NaN,No detecta Bad Buys o recall nulo
7,KNN,NaN,NaN,NaN,No detecta Bad Buys o recall nulo
